# 04 — Extension: Task-Specific Distillation

**Branch:** `extension/task-distillation`  
**Owner:** Person D

## Motivation

The original DistilBERT paper applies knowledge distillation during **pre-training**.  
This is expensive: it still requires ~90 hours on 8 V100s.

**Our extension asks:** Can we get comparable compression by distilling only during **fine-tuning**?

We train a fine-tuned BERT-base as the teacher, then train a smaller 3-layer student
using the teacher's soft probability outputs as supervision — no pre-training distillation at all.

## What we compare

| Model | Distillation | Layers | Params |
|-------|-------------|--------|--------|
| BERT-base | None | 12 | 110M |
| DistilBERT | Pre-training | 6 | 66M |
| Our student | Fine-tuning only | 3 | ~33M |

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertConfig,
    BertForSequenceClassification,
)
from datasets import load_dataset
from tqdm import tqdm
import numpy as np

from utils import print_model_info, measure_inference_speed, save_results, count_parameters, model_size_mb, load_results

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
TEACHER_MODEL  = 'bert-base-uncased'
MAX_LENGTH     = 128
BATCH_SIZE     = 32
STUDENT_LAYERS = 3       # Our student: 3 transformer layers
EPOCHS         = 5       # Train longer to compensate for fewer layers
LR             = 3e-5
TEMPERATURE    = 4.0     # Softmax temperature for soft labels (paper uses T=4)
ALPHA          = 0.5     # Balance between distillation loss and hard label loss
SEED           = 42

torch.manual_seed(SEED)

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────────
raw = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH, padding='max_length')

tokenized = raw.map(tokenize, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])
tokenized = tokenized.rename_column('label', 'labels')

train_loader = DataLoader(tokenized['train'],     batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE)

In [ ]:
# ── Step 1: Load fine-tuned teacher (from notebook 01 results) ─────────────
# If you saved the teacher model weights, load them here.
# Otherwise, re-fine-tune quickly (3 epochs).

from train import train_model

teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL, num_labels=2)
print('Fine-tuning teacher (BERT-base)...')
train_model(teacher, train_loader, val_loader, DEVICE, epochs=3, lr=2e-5)
teacher.eval()
print_model_info(teacher, 'Teacher (BERT-base)')

In [ ]:
# ── Step 2: Build student (3-layer BERT) ───────────────────────────────────
# Initialize from BERT-base config but with fewer layers.
# Copy weights from teacher layers 0, 5, 11 (evenly spaced — same strategy as DistilBERT).

teacher_config = teacher.config
student_config = BertConfig(
    vocab_size=teacher_config.vocab_size,
    hidden_size=teacher_config.hidden_size,          # keep same width (768)
    num_hidden_layers=STUDENT_LAYERS,
    num_attention_heads=teacher_config.num_attention_heads,
    intermediate_size=teacher_config.intermediate_size,
    num_labels=2,
)

student = BertForSequenceClassification(student_config)

# Copy embeddings from teacher
student.bert.embeddings.load_state_dict(teacher.bert.embeddings.state_dict())

# Copy evenly-spaced layers: layers 0, 5, 11 of the teacher → layers 0, 1, 2 of student
teacher_layers_to_copy = [0, 5, 11]
for student_idx, teacher_idx in enumerate(teacher_layers_to_copy):
    student.bert.encoder.layer[student_idx].load_state_dict(
        teacher.bert.encoder.layer[teacher_idx].state_dict()
    )

print_model_info(student, f'Student ({STUDENT_LAYERS}-layer BERT)')

In [ ]:
# ── Step 3: Task-specific distillation training loop ──────────────────────

def distillation_loss(student_logits, teacher_logits, labels, temperature, alpha):
    """
    Combined loss:
      alpha     * KL-divergence between soft student and teacher outputs (distillation)
      (1-alpha) * cross-entropy against hard labels
    """
    # Soft targets from teacher
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    kl_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (temperature ** 2)

    # Hard label loss
    ce_loss = F.cross_entropy(student_logits, labels)

    return alpha * kl_loss + (1 - alpha) * ce_loss


optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)
student.to(DEVICE)
teacher.to(DEVICE)

history = []

for epoch in range(EPOCHS):
    student.train()
    teacher.eval()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        token_type_ids = batch['token_type_ids'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        with torch.no_grad():
            teacher_out = teacher(input_ids=input_ids,
                                  attention_mask=attention_mask,
                                  token_type_ids=token_type_ids)

        student_out = student(input_ids=input_ids,
                              attention_mask=attention_mask,
                              token_type_ids=token_type_ids)

        loss = distillation_loss(
            student_out.logits, teacher_out.logits, labels, TEMPERATURE, ALPHA
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    # Validation
    student.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels_cpu = batch['labels']
            out = student(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                token_type_ids=batch['token_type_ids'].to(DEVICE),
            )
            preds = torch.argmax(out.logits, dim=-1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels_cpu.numpy())

    val_acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}: loss={avg_loss:.4f}, val_acc={val_acc:.2f}%')
    history.append({'epoch': epoch+1, 'train_loss': round(avg_loss, 4), 'val_accuracy': round(val_acc, 2)})

In [ ]:
# ── Compare all three models ───────────────────────────────────────────────
import pandas as pd

bert_results      = load_results('bert_baseline.json')
distilbert_results = load_results('distilbert.json')

student_speed = measure_inference_speed(student, val_loader, DEVICE)
student_params, _ = count_parameters(student)

comparison = pd.DataFrame([
    {'Model': 'BERT-base',         'Accuracy (%)': bert_results['val_accuracy'],
     'Params (M)': round(bert_results['total_parameters']/1e6, 1),
     'Distillation': 'None',       'Layers': 12},
    {'Model': 'DistilBERT',        'Accuracy (%)': distilbert_results['val_accuracy'],
     'Params (M)': round(distilbert_results['total_parameters']/1e6, 1),
     'Distillation': 'Pre-training', 'Layers': 6},
    {'Model': 'Our student (3L)',   'Accuracy (%)': round(history[-1]['val_accuracy'], 2),
     'Params (M)': round(student_params/1e6, 1),
     'Distillation': 'Fine-tuning only', 'Layers': STUDENT_LAYERS},
])

print(comparison.to_string(index=False))

In [ ]:
# ── Save extension results ─────────────────────────────────────────────────
extension_results = {
    'student_layers':     STUDENT_LAYERS,
    'temperature':        TEMPERATURE,
    'alpha':              ALPHA,
    'val_accuracy':       round(history[-1]['val_accuracy'], 2),
    'total_parameters':   student_params,
    'model_size_mb':      model_size_mb(student),
    'inference_speed':    student_speed,
    'training_history':   history,
}

save_results(extension_results, 'extension.json')
print(extension_results)